# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step walkthrough for loading and exploring the [FAIR^2 dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and includes record sets, fields, and protein quantification data from extracellular vesicles in human mast cells.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# View basic information about the dataset
meta = dataset.metadata
print(meta.name + ':')
print(meta.description)

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities in Croissant are referenced by their `@id` (unique identifier)—including record sets and fields. This ensures that we always reference the canonical, unambiguous entity.

In [ ]:
# Record set overview (list all record sets and their fields by @id)
print('Available Record Sets:')
for rs in dataset.metadata.record_sets:
    print(f"  @id: {rs.id}, name: '{rs.name}'")
    print('    Fields:')
    for field in rs.fields:
        print(f"      @id: {field.id}: {field.name}")

## 3. Data Extraction
Let's pick a record set and load its data into a DataFrame for analysis. All references use `@id` fields, as shown above.

In [ ]:
# Select a target record set (@id) for extraction
# Get all record set @id values dynamically, in case of schema update
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print('Available record set @id(s):', record_set_ids)

# We'll use the first record set for demonstration:
main_record_set_id = record_set_ids[0]
print(f'Selected record set: {main_record_set_id}')

# Extract all records from this record set
main_records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(main_records)
print('Fields (columns) as @id:')
print(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using field `@id` as column labels. We'll:
- Filter records by a numeric field
- Normalize the field
- (If available) group by a categorical field

First, let's list available fields and types for our selected record set.

In [ ]:
# List field @id and types for reference
selected_rs = [rs for rs in dataset.metadata.record_sets if rs.id == main_record_set_id][0]
print('Fields for selected record set:')
for field in selected_rs.fields:
    print(f"@id: {field.id}, name: {field.name}, dataType: {field.data_type}")

In [ ]:
# Select a numeric field for filtering (example: molecular_weight or another shown in field listing)
# Let's try auto-detecting a float/integer field
numeric_field_id = None
for field in selected_rs.fields:
    if field.data_type and ('Float' in field.data_type or 'Integer' in field.data_type or 'Number' in str(field.data_type)):
        if field.id in df.columns:
            numeric_field_id = field.id
            break
if not numeric_field_id:
    raise Exception("No numeric field found for EDA!")
print('Selected numeric field @id:', numeric_field_id)

# Ensure the column is numeric type
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter for meaningful entries (e.g., values > threshold)
threshold = df[numeric_field_id].mean()  # or use a fixed threshold like 10000
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field (e.g., a string field that's not the numeric one)
group_field_id = None
for field in selected_rs.fields:
    if field.id != numeric_field_id and field.data_type and 'Text' in str(field.data_type) and field.id in df.columns:
        group_field_id = field.id
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print('No suitable categorical group field found for this record set.')

## 5. Visualization
Visualize data distributions and relationships between fields using matplotlib. We'll show a histogram of the selected numeric field and, if available, a mean by group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id:
    group_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10, 4))
    group_means.plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field_id} (Top 10)")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
Using `mlcroissant`, we've loaded and explored the FAIR^2 dataset of mass spectrometry analysis of extracellular vesicle proteins from human mast cells. We used Croissant `@id` references to manage record sets and fields, loaded main data into a DataFrame, filtered and normalized a key quantitative field, optionally grouped by a categorical attribute, and visualized the data distribution.

This schema-driven exploration ensures full reproducibility and auditability. For advanced analysis, iterate over the field and record set `@id`s as needed!